## EJERCICIO 1

In [1]:
import numpy as np
import random
import copy
import csv
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# INDIVIDUO BINARIO
class IndividuoBinarioFunciones:
    def __init__(self, limites, num_bits=16):
        self.limites = limites
        self.num_bits = num_bits
        self.fitness = None
        # cadena binaria
        self.genotipo = [np.random.choice([0, 1], size=num_bits) for _ in limites]

    def mutar(self, tasa_mutacion):
        for cromosoma in self.genotipo:
            for i in range(self.num_bits):
                if random.random() < tasa_mutacion:
                    cromosoma[i] = 1 - cromosoma[i]
                    break

    def cruzar(self, madre):
        hijo1 = copy.deepcopy(self)
        hijo2 = copy.deepcopy(madre)
        
        for i in range(len(self.genotipo)):
            punto_cruza = random.randint(1, self.num_bits - 2)
            hijo1.genotipo[i] = np.concatenate((self.genotipo[i][:punto_cruza], madre.genotipo[i][punto_cruza:]))
            hijo2.genotipo[i] = np.concatenate((madre.genotipo[i][:punto_cruza], self.genotipo[i][punto_cruza:]))
            
        return hijo1, hijo2

    def decodificar(self):
        # Genotipo a Fenotipo
        valores_reales = []
        for i, cromosoma in enumerate(self.genotipo):
            lim_inf, lim_sup = self.limites[i]
            
            # base 2 a base 10
            entero = 0
            for bit in cromosoma:
                entero = (entero * 2) + bit
                
            max_entero = (2 ** self.num_bits) - 1
            valor_real = lim_inf + (entero / max_entero) * (lim_sup - lim_inf)
            valores_reales.append(valor_real)
            
        return valores_reales

# GENETICO BINARIO
class AlgoritmoGeneticoBinarioFunciones:
    def __init__(self, funcion_objetivo, limites, num_bits=16, tam_poblacion=50, generaciones=100, tasa_mutacion=0.01):
        self.funcion_objetivo = funcion_objetivo
        self.limites = limites
        self.num_bits = num_bits
        self.tam_poblacion = tam_poblacion
        self.generaciones = generaciones
        self.tasa_mutacion = tasa_mutacion
        self.poblacion = []
        self.mejor_individuo = None

    def evaluar(self, individuo):
        # Decodificar genotipo a fenotipo
        valores_reales = individuo.decodificar()
        individuo.fitness = self.funcion_objetivo(valores_reales)

        # Elitismo
        if self.mejor_individuo is None or individuo.fitness < self.mejor_individuo.fitness:
            self.mejor_individuo = copy.deepcopy(individuo)

    def inicializar_poblacion(self):
        self.poblacion = [IndividuoBinarioFunciones(self.limites, self.num_bits) for _ in range(self.tam_poblacion)]
        for ind in self.poblacion:
            self.evaluar(ind)

    def seleccion_comp(self, k=3):
        competencia = random.sample(self.poblacion, k)
        def obtener_fitness(ind):
                return ind.fitness
        return min(competencia, key=obtener_fitness)

    def evolucionar(self):
        nueva_poblacion = []
        nueva_poblacion.append(copy.deepcopy(self.mejor_individuo))

        while len(nueva_poblacion) < self.tam_poblacion:
            padre = self.seleccion_comp()
            madre = self.seleccion_comp()
            
            hijo1, hijo2 = padre.cruzar(madre)
            hijo1.mutar(self.tasa_mutacion)
            hijo2.mutar(self.tasa_mutacion)
            
            self.evaluar(hijo1)
            self.evaluar(hijo2)
            
            nueva_poblacion.extend([hijo1, hijo2])
            
        self.poblacion = nueva_poblacion[:self.tam_poblacion]

    def ejecutar(self):
        self.inicializar_poblacion()
        for _ in tqdm(range(self.generaciones)):
            self.evolucionar()
        return self.mejor_individuo


# GRADIENTE DESCENDENTE
def gradiente_descendente(funcion, x0, limites, cte=0.01, iteraciones=1000):
    x = np.array(x0)
    h = 1e-5
    
    for _ in range(iteraciones):
        grad = np.zeros_like(x)

        # Diferencias finitas
        for i in range(len(x)):
            xh = x.copy()
            xh[i] += h
            grad[i] = (funcion(xh) - funcion(x)) / h
            
        x = x - cte * grad
            
    return x, funcion(x)


def f1(x):
    val = x[0]
    return -val * np.sin(np.sqrt(np.abs(val)))

def f2(x):
    x_val, y_val = x[0], x[1]
    term1 = (x_val**2 + y_val**2)**0.25
    term2 = np.sin(50 * (x_val**2 + y_val**2)**0.1)**2 + 1
    return term1 * term2


if __name__ == "__main__":
    print("--- Optimizando Función i ---")
    ag1 = AlgoritmoGeneticoBinarioFunciones(f1, limites=[(-512, 512)], tam_poblacion=100, generaciones=150, tasa_mutacion=0.05)
    mejor_ag1 = ag1.ejecutar()
    reales_ag1 = mejor_ag1.decodificar()
    print(f"AG -> Mínimo en x={reales_ag1[0]:.4f} | f(x)={mejor_ag1.fitness:.4f}")
    
    punto_inicio_gd = [0.0] 
    x_gd1, f_gd1 = gradiente_descendente(f1, punto_inicio_gd, limites=[(-512, 512)])
    print(f"GD -> Mínimo en x={x_gd1[0]:.4f} | f(x)={f_gd1:.4f}\n")


    print("--- Optimizando Función ii ---")
    ag2 = AlgoritmoGeneticoBinarioFunciones(f2, limites=[(-100, 100), (-100, 100)], tam_poblacion=150, generaciones=200, tasa_mutacion=0.05)
    mejor_ag2 = ag2.ejecutar()
    reales_ag2 = mejor_ag2.decodificar()
    print(f"AG -> Mínimo en x={reales_ag2[0]:.4f}, y={reales_ag2[1]:.4f} | f(x,y)={mejor_ag2.fitness:.4f}")
    
    punto_inicio_gd2 = [50.0, 50.0]
    x_gd2, f_gd2 = gradiente_descendente(f2, punto_inicio_gd2, limites=[(-100, 100), (-100, 100)], cte=0.001)
    print(f"GD -> Mínimo en x={x_gd2[0]:.4f}, y={x_gd2[1]:.4f} | f(x,y)={f_gd2:.4f}")

--- Optimizando Función i ---


100%|██████████| 150/150 [00:00<00:00, 305.53it/s]


AG -> Mínimo en x=421.0142 | f(x)=-418.9826
GD -> Mínimo en x=5.0801 | f(x)=-3.9402

--- Optimizando Función ii ---


100%|██████████| 200/200 [00:01<00:00, 152.43it/s]

AG -> Mínimo en x=0.0015, y=-0.0015 | f(x,y)=0.0818
GD -> Mínimo en x=48.4170, y=48.4170 | f(x,y)=8.5797
